In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [1]:
import torch
import transformers
import datasets
import bitsandbytes
import accelerate
import peft
import trl
import unsloth


torch.__version__, transformers.__version__, datasets.__version__, bitsandbytes.__version__, accelerate.__version__, peft.__version__, trl.__version__

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


('2.5.1+cu124', '4.48.3', '3.3.0', '0.45.2', '1.3.0', '0.14.0', '0.15.0')

In [3]:
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # 4bit quantization to reduce memory usage

# Get the model

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.12: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [5]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,)

# Set LoRa adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,  # LoRa rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,  # Rank Stabilized LoRA
    loftq_config=None,
)

Unsloth 2025.2.12 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

# Get the dataset

In [8]:
from datasets import load_dataset

dataset = load_dataset("Tom158/Nutrinition_1k_LLama3", split = "train")

README.md:   0%|          | 0.00/525 [00:00<?, ?B/s]

(…)-00000-of-00001-bb0de1c31c25a21a.parquet:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [20]:
dataset.column_names

['System', 'Conversation', 'text']

In [10]:
dataset[0]["System"]

'You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.'

In [36]:
dataset[1]["System"]

'You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.'

In [11]:
dataset[0]["Conversation"]

'## User: Hi there! I\'m looking for some nutritional advice.\n## Nutritionist: Hello! How can I help you today?\n## User: I\'ve been trying to eat healthier and I was wondering if you could tell me about collard greens. Specifically, what\'s in them and how healthy they are.\n## Nutritionist: Collard greens, also known as collards, are a nutrient-rich leafy green vegetable. They\'re very low in calories, with only 32 per serving. They\'re also very high in fiber, with 4 grams per serving, which can help support digestive health.\n\nLet\'s take a look at the nutritional details you provided: total fat is relatively low, with only 0.6g, and it\'s mostly unsaturated. There\'s no cholesterol, sodium is quite low at 17mg, and there are only 0.46g of sugars per serving. Protein content is moderate, at 3.02g.\n\nCollards are also an excellent source of vitamins A (5019 IU), C (35.3mg), E (2.26mg), K (437.1mcg), and B6 (0.165mg). They\'re a good source of folate (129mcg) as well. In terms of 

In [12]:
dataset[0]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.<|eot_id|><|start_header_id|>user<|end_header_id|><|eot_id|> Hi there! I\'m looking for some nutritional advice.<|start_header_id|>assistant<|end_header_id|>Hello! How can I help you today?<|eot_id|><|end_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.<|eot_id|><|start_header_id|>user<|end_header_id|><|eot_id|> I\'ve been trying to eat healthier and I was wondering if you could tell me about collard greens. Specifically, what\'s in them and how healthy they

# Convert the dataset into conversations

In [54]:
def convert_format(example):
    messages = []

    system_message = example["System"].strip()
    messages.append({"role": "system", "content": system_message})

    text = example["text"].strip()

    # Split by special tokens
    turns = text.split("<|eot_id|>")
    role_map = {"system": "system", "user": "user", "assistant": "assistant"}

    for turn in turns:
        turn = turn.strip()
        if not turn:
            continue

        if "<|start_header_id|>" in turn and "<|end_header_id|>" in turn:
            role_start = turn.find("<|start_header_id|>") + len("<|start_header_id|>")
            role_end = turn.find("<|end_header_id|>")
            role = turn[role_start:role_end].strip()

            content_start = role_end + len("<|end_header_id|>")
            content = turn[content_start:].strip()

            if role in role_map and content:
                messages.append({"role": role_map[role], "content": content})

    return {"conversations": messages}

d = dataset.map(convert_format)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [62]:
d[10]["conversations"]

[{'content': 'You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.',
  'role': 'system'},
 {'content': 'You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.',
  'role': 'system'},
 {'content': 'Hello! How can I assist you today?', 'role': 'assistant'},
 {'content': 'You serve as a professional nutrition advisor and a friendly assistant. When the user seeks nutrition-related advice, greet them warmly and engage in small talk if appropriate. Ensure that interactions are natural and human-like.',
  'role': 'system'},
 {'content': "Thanks for sharing that information. Given your concerns with high blood pressure, it's essential to focus on re